In [13]:
import torch
import numpy as np

from dinosaw.helpers import add_custom_font, get_model, get_models, get_features, ModelTypes, model_names #, get_features
from dinosaw.models.vit_wrapper import PretrainedViTWrapper, MODEL_LIST
from dinosaw.utils import do_2D_pca
from skimage.transform import resize

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import Literal, TypeAlias

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [2]:
selected_model = 'alibi_dv2_coco'
selected_models: tuple[ModelTypes, ...] = ('dv2', 'dv3', 'alibi_dv2_coco')

models = get_models(selected_models, '../../trained_models', device=DEVICE,  conf_path='../../dinov3')
S = models[selected_models[0]].stride

n_dims = 384

In [9]:
image_names = ['labradors', 'new_york', 'bimodal']

shortest_side = 683
longest_side = 1024

images = []

for image_name in image_names:
    image = Image.open(f'data/more_pcas/{image_name}.png')
    ih, iw = image.size[::-1]
    scale = shortest_side / min(ih, iw)
    image = image.resize((int(iw * scale), int(ih * scale)))
    
    ox, oy = (image.size[0] - longest_side) // 2, (image.size[1] - shortest_side) // 2
    image = image.crop((ox, oy, ox + longest_side, oy + shortest_side))

    images.append(image)


In [ ]:
features: dict[ModelTypes, list[np.ndarray]] = {key: [] for key in selected_models}

for key, model in models.items():
    for image in images:
        feats = get_features(model, image, device=DEVICE)
        reduced = do_2D_pca(feats, 3, post_norm='minmax')[:, :, 0:3]
        features[key].append(reduced)

In [30]:
def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

In [40]:
%%capture
add_custom_font('resources/fonts', 'Grotesk')
n_rows, n_cols = len(selected_models) + 1, len(images)
FS = 24
W, H = 4.75, 3

titles = ["Labradors", "New York", "Bimodal cathode"]
fig, axs = plt.subplots(n_rows, n_cols, figsize=(W * n_cols, H * n_rows))

for col, image in enumerate(images):
    axs[0, col].imshow(image, rasterized=True)
    hide_axes(axs[0, col])
    axs[0, col].set_title(titles[col], fontsize=FS, weight=500)

    for row, key in enumerate(selected_models):
        ax = axs[row + 1, col]
        ax.imshow(features[key][col], rasterized=True)
        hide_axes(ax)

        if col == 0:
            weight = 700 if 'alibi' in key else 500
            name = model_names[key]
            name = name.replace('(COCO)', '')
            ax.set_ylabel(name, fontsize=FS, weight=weight)

plt.tight_layout()
plt.savefig('saved/08.jpeg', dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})